# Assignment 4: Autoencoder for Anomaly Detection (ECG dataset)
**Aim:** Use an autoencoder to detect abnormal ECG signals.

**Idea:** Train the autoencoder **only on normal ECGs**. It learns to reconstruct normal signals well.
An abnormal ECG gives a **high reconstruction error** → if error > threshold it is an anomaly.

Dataset: 4998 ECGs, 140 values each; column 140 is the label (1 = normal, 0 = abnormal).

## a. Import required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

I0000 00:00:1793640476.074415   32528 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640477.284146   32528 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## b. Upload / access the dataset

In [2]:
df = pd.read_csv("dataset/ecg-csv/ecg.csv", header=None)
print(df.shape)
df.head()

(4998, 141)


,0,1,2,3,4,5,6,7,8,9,...,131,132,133,134,135,136,137,138,139,140
0,-0.112522,-2.827204,-3.773897,-4.349751,-4.376041,-3.474986,-2.181408,-1.818286,-1.250522,-0.477492,...,0.792168,0.933541,0.796958,0.578621,0.257740,0.228077,0.123431,0.925286,0.193137,1.0
1,-1.100878,-3.996840,-4.285843,-4.506579,-4.022377,-3.234368,-1.566126,-0.992258,-0.754680,0.042321,...,0.538356,0.656881,0.787490,0.724046,0.555784,0.476333,0.773820,1.119621,-1.436250,1.0
2,-0.567088,-2.593450,-3.874230,-4.584095,-4.187449,-3.151462,-1.742940,-1.490659,-1.183580,-0.394229,...,0.886073,0.531452,0.311377,-0.021919,-0.713683,-0.532197,0.321097,0.904227,-0.421797,1.0
3,0.490473,-1.914407,-3.616364,-4.318823,-4.268016,-3.881110,-2.993280,-1.671131,-1.333884,-0.965629,...,0.350816,0.499111,0.600345,0.842069,0.952074,0.990133,1.086798,1.403011,-0.383564,1.0
4,0.800232,-0.874252,-2.384761,-3.973292,-4.338224,-3.802422,-2.534510,-1.783423,-1.594450,-0.753199,...,1.148884,0.958434,1.059025,1.371682,1.277392,0.960304,0.971020,1.614392,1.421456,1.0


In [3]:
X = df.iloc[:, :-1].values          # 140 ECG values
y = df.iloc[:, -1].values.astype(int)  # 1 = normal, 0 = abnormal
print("Normal:", (y == 1).sum(), " Abnormal:", (y == 0).sum())

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Train only on NORMAL ECGs
x_train_normal = x_train[y_train == 1]
print("Training on", x_train_normal.shape[0], "normal ECGs")

Normal: 2919  Abnormal: 2079
Training on 2335 normal ECGs


In [4]:
plt.figure(figsize=(12, 4))
plt.plot(x_train[y_train == 1][0], label="Normal ECG")
plt.plot(x_train[y_train == 0][0], label="Abnormal ECG")
plt.legend(); plt.title("Sample ECG signals"); plt.show()

/tmp/ipykernel_32528/3442217877.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.title("Sample ECG signals"); plt.show()


## c. Encoder: converts input into latent representation (140 → 32 → 16 → 8)
## d. Decoder: converts it back to the original input (8 → 16 → 32 → 140)

In [5]:
encoder = keras.Sequential([
    keras.Input(shape=(140,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),        # latent representation
], name="encoder")

decoder = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(140, activation="sigmoid"),   # output in 0-1 (MinMax scaled)
], name="decoder")

autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.summary()

Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Sequential)            │ (None, 8)              │         5,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 140)            │         5,308 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,484 (40.95 KB)

 Trainable params: 10,484 (40.95 KB)

 Non-trainable params: 0 (0.00 B)

## e. Compile the model with Optimizer, Loss and Evaluation Metrics

In [6]:
autoencoder.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = autoencoder.fit(x_train_normal, x_train_normal,   # input = target
                          epochs=30, batch_size=64,
                          validation_data=(x_test, x_test), shuffle=True)

Epoch 1/30


 1/37 ━━━━━━━━━━━━━━━━━━━━ 20s 564ms/step - loss: 0.0270 - mae: 0.1357

36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0203 - mae: 0.1126   

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0201 - mae: 0.1122

37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0201 - mae: 0.1122 - val_loss: 0.0163 - val_mae: 0.0953


Epoch 2/30


... (87 outputs left out)

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0015 - mae: 0.0255 - val_loss: 0.0064 - val_mae: 0.0487


Epoch 30/30


 1/37 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0017 - mae: 0.0267

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0015 - mae: 0.0254 - val_loss: 0.0064 - val_mae: 0.0486


In [7]:
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title("Autoencoder training loss"); plt.legend()
plt.show()

/tmp/ipykernel_32528/3738231248.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Anomaly detection using reconstruction error

In [8]:
# Threshold = mean + 1 std of reconstruction error on normal training data
train_recon = autoencoder.predict(x_train_normal, verbose=0)
train_loss = np.mean(np.square(x_train_normal - train_recon), axis=1)
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold:", threshold)

test_recon = autoencoder.predict(x_test, verbose=0)
test_loss = np.mean(np.square(x_test - test_recon), axis=1)

# error > threshold -> anomaly (0), else normal (1)
y_pred = np.where(test_loss > threshold, 0, 1)
print("Anomalies detected:", (y_pred == 0).sum())

Threshold: 0.0035635645233468346
Anomalies detected: 461


In [9]:
plt.figure(figsize=(10, 4))
plt.hist(test_loss[y_test == 1], bins=50, alpha=0.6, label="Normal")
plt.hist(test_loss[y_test == 0], bins=50, alpha=0.6, label="Abnormal")
plt.axvline(threshold, color="r", linestyle="--", label="Threshold")
plt.xlabel("Reconstruction error"); plt.ylabel("Count"); plt.legend()
plt.title("Reconstruction Error Distribution"); plt.show()

/tmp/ipykernel_32528/3405663472.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Reconstruction Error Distribution"); plt.show()


## Evaluate the model

In [10]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Abnormal", "Normal"]))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Abnormal", "Normal"], yticklabels=["Abnormal", "Normal"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()

Accuracy : 0.955
Precision: 1.0
Recall   : 0.922945205479452
              precision    recall  f1-score   support

    Abnormal       0.90      1.00      0.95       416
      Normal       1.00      0.92      0.96       584

    accuracy                           0.95      1000
   macro avg       0.95      0.96      0.95      1000
weighted avg       0.96      0.95      0.96      1000



/tmp/ipykernel_32528/1890165373.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()


In [11]:
# Original vs reconstructed signal
for label, name in [(1, "Normal"), (0, "Abnormal")]:
    i = np.where(y_test == label)[0][0]
    plt.figure(figsize=(10, 3))
    plt.plot(x_test[i], label="Original")
    plt.plot(test_recon[i], label="Reconstructed")
    plt.fill_between(range(140), x_test[i], test_recon[i], color="lightcoral", label="Error")
    plt.title(f"{name} ECG (error = {test_loss[i]:.4f})"); plt.legend(); plt.show()

/tmp/ipykernel_32528/2133566856.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f"{name} ECG (error = {test_loss[i]:.4f})"); plt.legend(); plt.show()
